# Relation Extraction Notebook

**NLP IE — Detect relations like person-works-for-company**

## Project Overview

Extract **subject-predicate-object** triples from text using
spaCy NER and verb pattern matching.

## Learning Objectives

1. Use NER + verb patterns for relation extraction.
2. Extract structured knowledge triples.
3. Understand limitations of rule-based RE.

## Problem Statement

Extract who works for whom, who founded what, etc.

## Why This Project Matters

- Knowledge graphs power search and Q&A.
- RE automates knowledge base construction.

## Dataset Overview

12 sentences with known relations.

## Dataset Source and License Notes

Synthetic.

## Environment Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ["pandas","numpy","matplotlib","seaborn","spacy"]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
try:
    import spacy; spacy.load("en_core_web_sm")
except OSError:
    subprocess.check_call([sys.executable,"-m","spacy","download","en_core_web_sm"])
print("Environment ready.")


## Imports

In [ ]:
import re, warnings
from collections import defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import spacy
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
%matplotlib inline
nlp = spacy.load("en_core_web_sm")
SEED = 42; np.random.seed(SEED)
print("Loaded.")


## Configuration / Constants

In [ ]:
WORKS = {"work","works","worked","join","joined"}
FOUNDED = {"founded","co-founded","started","created","established"}
LEADS = {"leads","led","manages","heads","directs","runs"}
print("Verb sets configured.")


## Dataset Download or Loading

In [ ]:
SENTENCES = [
    {"text":"Tim Cook works at Apple as the CEO.","expected":("Tim Cook","works-at","Apple")},
    {"text":"Satya Nadella joined Microsoft in 1992.","expected":("Satya Nadella","works-at","Microsoft")},
    {"text":"Elon Musk founded SpaceX in 2002.","expected":("Elon Musk","founded","SpaceX")},
    {"text":"Jeff Bezos started Amazon from his garage.","expected":("Jeff Bezos","founded","Amazon")},
    {"text":"Sundar Pichai leads Google as CEO.","expected":("Sundar Pichai","leads","Google")},
    {"text":"Jensen Huang co-founded NVIDIA in 1993.","expected":("Jensen Huang","founded","NVIDIA")},
    {"text":"Mark Zuckerberg created Facebook.","expected":("Mark Zuckerberg","founded","Facebook")},
    {"text":"Lisa Su heads AMD.","expected":("Lisa Su","leads","AMD")},
    {"text":"Sam Altman leads OpenAI.","expected":("Sam Altman","leads","OpenAI")},
    {"text":"The weather is nice today.","expected":None},
    {"text":"Python is a popular language.","expected":None},
]
df = pd.DataFrame(SENTENCES)
print(f"Loaded {len(df)} sentences")


## Data Validation Checks

In [ ]:
assert df["text"].notna().all()
print(f"Validated: {len(df)}")


## Exploratory Data Analysis

In [ ]:
has = df["expected"].notna().sum()
print(f"With relations: {has}, without: {len(df)-has}")


## Target Analysis

Subject-predicate-object triples.

## Train/Validation/Test Split Strategy

Not applicable — rule-based.

## Preprocessing Strategy

NER for entities, verb patterns for relation type.

## Feature Engineering — Relation Extractor

In [ ]:
def extract_relations(text):
    doc = nlp(text)
    relations = []
    persons = [e for e in doc.ents if e.label_ == "PERSON"]
    orgs = [e for e in doc.ents if e.label_ == "ORG"]
    for person in persons:
        for org in orgs:
            t = text.lower()
            if any(v in t for v in FOUNDED):
                relations.append((person.text, "founded", org.text))
            elif any(v in t for v in LEADS):
                relations.append((person.text, "leads", org.text))
            elif any(v in t for v in WORKS):
                relations.append((person.text, "works-at", org.text))
    return relations
print("extract_relations() ready.")


## Baseline Model — Extract Relations

In [ ]:
results = []
for _, row in df.iterrows():
    rels = extract_relations(row["text"])
    results.append(rels)
    if rels:
        print(f"  {row["text"][:50]}...")
        for r in rels: print(f"    -> {r[0]} [{r[1]}] {r[2]}")
df["extracted"] = results


## LazyPredict Benchmark

> **Not applicable.** NLP IE task.

## FLAML AutoML Run

> **Not applicable.** FLAML not used for NLP IE.

## Additional Best-Library Workflow — Evaluation

In [ ]:
correct = 0; total_exp = 0
for _, row in df.iterrows():
    if row["expected"] is not None:
        total_exp += 1
        if any(e[1] == row["expected"][1] for e in row["extracted"]):
            correct += 1; print(f"  ✓ {row["expected"]}")
        else:
            print(f"  ✗ Expected {row["expected"]}, got {row["extracted"]}")
print(f"\nAccuracy: {correct/total_exp:.1%}")


## Top 3 Model Selection

Single rule-based pipeline.

In [ ]:
all_rels = [r for rels in results for r in rels]
types = pd.Series([r[1] for r in all_rels]).value_counts()
fig, ax = plt.subplots(figsize=(7,4))
types.plot.bar(ax=ax, color="steelblue")
ax.set_title("Relation Types"); plt.tight_layout(); plt.show()


## Final Training and Evaluation of Top 3

Dependency-based extractor is the final system.

## Error Analysis

In [ ]:
for _, row in df.iterrows():
    if row["expected"] is None and row["extracted"]:
        print(f"  False positive: {row["text"][:50]}")


## Interpretation and Business Insight

1. NER + verb patterns capture common relations.
2. No-relation sentences correctly filtered.
3. Real text is more complex.

## Limitations

- Limited relation types.
- No coreference.
- Depends on NER accuracy.

## How to Improve This Project

1. Transformer-based RE.
2. Add coreference.
3. Build knowledge graph.

## Production Considerations

- Store in graph database.
- Handle disambiguation.
- Monitor accuracy.

## Common Mistakes

1. Ignoring negation.
2. Not handling coreference.
3. Spurious relation extraction.

## Mini Challenge / Exercises

1. Add acquired/invested relations.
2. Build a knowledge graph.
3. Add temporal extraction.

## Final Summary / Key Takeaways

1. RE turns text into structured knowledge.
2. Verb patterns capture common relations.
3. For production, use transformer models.
4. Knowledge graphs are the natural output.